In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS gold;")

# 1.1 Dimensões
df_info = spark.read.table("silver.tb_info_filmes")
df_dim_movies = df_info.select(
    F.col("id_filme").cast("string"), "titulo", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse"
).withColumn("sk_movie_id", F.monotonically_increasing_id())
df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")

df_gen = spark.read.table("silver.tb_generos").select("nome_genero").distinct()
df_dim_genres = df_gen.withColumn("sk_genre_id", F.monotonically_increasing_id())
df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")

df_pessoas = spark.read.table("silver.tb_pessoas_empresas").filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
df_dim_people = df_pessoas.select("nome_entidade", "tipo_entidade").distinct() \
    .withColumnRenamed("nome_entidade", "nome_pessoa") \
    .withColumnRenamed("tipo_entidade", "tipo_pessoa") \
    .withColumn("sk_person_id", F.monotonically_increasing_id())
df_dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")

df_comp = spark.read.table("silver.tb_pessoas_empresas").filter(F.col("tipo_entidade") == "Produtora")
df_dim_companies = df_comp.select("nome_entidade").distinct() \
    .withColumnRenamed("nome_entidade", "nome_produtora") \
    .withColumn("sk_company_id", F.monotonically_increasing_id())
df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")

df_rev = spark.read.table("silver.tb_avaliacoes_usuarios")
df_rev_agg = df_rev.groupBy("id_filme").agg(
    F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
    F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
)
df_dim_reviews = df_rev_agg.join(df_dim_movies, "id_filme", "inner") \
    .select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios") \
    .withColumn("sk_review_id", F.monotonically_increasing_id())
df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")

# 1.2 Tabelas Bridge
df_silver_gen = spark.read.table("silver.tb_generos")
df_bridge_genre = df_silver_gen.join(df_dim_movies, "id_filme", "inner") \
    .join(df_dim_genres, "nome_genero", "inner") \
    .select("sk_movie_id", "sk_genre_id")
df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")

df_silver_pessoas = spark.read.table("silver.tb_pessoas_empresas").filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
df_bridge_person = df_silver_pessoas.join(df_dim_movies, "id_filme", "inner") \
    .join(df_dim_people, (df_silver_pessoas.nome_entidade == df_dim_people.nome_pessoa) & (df_silver_pessoas.tipo_entidade == df_dim_people.tipo_pessoa), "inner") \
    .select("sk_movie_id", "sk_person_id")
df_bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")

df_silver_comp = spark.read.table("silver.tb_pessoas_empresas").filter(F.col("tipo_entidade") == "Produtora")
df_bridge_comp = df_silver_comp.join(df_dim_movies, "id_filme", "inner") \
    .join(df_dim_companies, df_silver_comp.nome_entidade == df_dim_companies.nome_produtora, "inner") \
    .select("sk_movie_id", "sk_company_id")
df_bridge_comp.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_company")

# 1.3 Fato
df_fin = spark.read.table("silver.tb_financeiro_filmes")
df_met = spark.read.table("silver.tb_metricas_engajamento")
df_fact = df_dim_movies.select("sk_movie_id", "id_filme") \
    .join(df_fin, "id_filme", "left") \
    .join(df_met, "id_filme", "left") \
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
df_fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")

# 2. Tabela de Contexto GenAI
df_atores_agg = df_bridge_person.join(df_dim_people, "sk_person_id") \
    .filter(F.col("tipo_pessoa") == "Ator") \
    .groupBy("sk_movie_id").agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores_principais"))

df_diretores_agg = df_bridge_person.join(df_dim_people, "sk_person_id") \
    .filter(F.col("tipo_pessoa") == "Diretor") \
    .groupBy("sk_movie_id").agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("diretor"))

df_genai = df_dim_movies.select("sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse") \
    .join(df_fact.select("sk_movie_id", "receita_brl", "orcamento_brl"), "sk_movie_id", "left") \
    .join(df_atores_agg, "sk_movie_id", "left") \
    .join(df_diretores_agg, "sk_movie_id", "left")

df_genai = df_genai.withColumn("titulo_f", F.coalesce(F.col("titulo"), F.lit("Nome Desconhecido")))
df_genai = df_genai.withColumn("ano_f", F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano desconhecido")))
df_genai = df_genai.withColumn("receita_f", F.when(F.col("receita_brl").isNull(), F.lit("valor não informado")).otherwise(F.concat(F.lit("R$ "), F.col("receita_brl").cast("string"))))
df_genai = df_genai.withColumn("orcamento_f", F.when(F.col("orcamento_brl").isNull(), F.lit("valor não informado")).otherwise(F.concat(F.lit("R$ "), F.col("orcamento_brl").cast("string"))))
df_genai = df_genai.withColumn("atores_f", F.coalesce(F.col("atores_principais"), F.lit("elenco não informado")))
df_genai = df_genai.withColumn("diretor_f", F.coalesce(F.col("diretor"), F.lit("diretor não informado")))
df_genai = df_genai.withColumn("sinopse_f", F.coalesce(F.col("sinopse"), F.lit("Sinopse indisponível.")))

template = F.concat(
    F.lit("O filme "), F.col("titulo_f"),
    F.lit(", lançado no ano de "), F.col("ano_f"),
    F.lit(", faturou "), F.col("receita_f"),
    F.lit(" e teve um custo de "), F.col("orcamento_f"),
    F.lit(". Estrelado por "), F.col("atores_f"),
    F.lit(" e dirigido por "), F.col("diretor_f"),
    F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_f")
)

df_gold_genai = df_genai.select(F.col("id_filme").alias("movie_id"), F.col("titulo").alias("title"), template.alias("llm_context_document"))
df_gold_genai.write.format("delta").mode("overwrite").saveAsTable("gold.gold_genai_movies_context")
print("Camada Gold gerada com sucesso!")

Camada Gold gerada com sucesso!


In [0]:
%sql
SELECT SUM(receita_brl) AS receita_total_brl 
FROM gold.fact_movies_performance;

receita_total_brl
967629890188.48


In [0]:
%sql
SELECT d.titulo, f.popularidade
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
ORDER BY f.popularidade DESC NULLS LAST
LIMIT 5;

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


In [0]:
%sql
SELECT g.nome_genero, COUNT(b.sk_movie_id) AS qtd_filmes
FROM gold.dim_genres g
JOIN gold.bridge_movie_genre b ON g.sk_genre_id = b.sk_genre_id
GROUP BY g.nome_genero
ORDER BY qtd_filmes DESC;

nome_genero,qtd_filmes
/3bOPUh6ClrWNmMdAOrXYL6gMz61.jpg,30457
/9qY8wA2VafiwjJ3rV5mRR5yrT3n.jpg,18612
/mgexGVzgcVKth6Vs4i8CYbVxYCd.jpg,17348
The Stop-Motion Samurai Film,9424
/zMiekZdVchQAFPoatIzqkyCc3WR.jpg,9163
Teruko displays a mature charm and an adult-like sex appeal. In private,6996
"have finally uncovered the whereabouts of the Psycho Surgeons and get set to exact the bloodiest of revenge. But what does this mean for The Gore Collector himself? As he returns to where it all started - the Bunker of Blood - he will find out what this splatter-soaked road trip across a fevered nightmarescape has REALLY done to his mind AND body!""",5541
Popes,4320
finding an insightful lesson with each encounter. McNichols' message as a priest,4159
ya filthy animals!,3676


In [0]:
%sql
SELECT d.titulo, f.receita_usd, f.receita_brl,
       RANK() OVER(ORDER BY f.receita_usd DESC NULLS LAST) AS ranking
FROM gold.fact_movies_performance f
JOIN gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
WHERE f.receita_usd IS NOT NULL
ORDER BY ranking
LIMIT 10;

titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,14311080000.00,1
Avatar: The Way of Water,2320250281.00,11859031211.22,2
AVENGERS: INFINITY WAR,2052415039.00,10490098505.83,3
spider-man: no way home,1921847111.00,9822752769.03,4
The Lion King,1663075401.00,8500144682.05,5
Top Gun: Maverick,1488732821.00,7609062321.41,6
Barbie,1428545028.00,7301436492.61,7
The Super Mario Bros. Movie,1355725263.00,6929247391.72,8
Black Panther,1349926083.00,6899607202.82,9
Star Wars: The Last Jedi,1332698830.00,6811556990.01,10


In [0]:
%sql
WITH Limite AS (
    SELECT MAX(data_lancamento) AS max_data 
    FROM gold.dim_movies 
    WHERE data_lancamento <= current_date()
)
SELECT p.nome_pessoa, COUNT(b.sk_movie_id) AS qtd_participacoes
FROM gold.dim_people p
JOIN gold.bridge_movie_person b ON p.sk_person_id = b.sk_person_id
JOIN gold.dim_movies d ON b.sk_movie_id = d.sk_movie_id
CROSS JOIN Limite l
WHERE p.tipo_pessoa = 'Ator'
  AND d.data_lancamento >= add_months(l.max_data, -24)
  AND d.data_lancamento <= l.max_data
GROUP BY p.nome_pessoa
ORDER BY qtd_participacoes DESC
LIMIT 1;

nome_pessoa,qtd_participacoes
Ijon Stewart,167


In [0]:
%sql
WITH Limite AS (
    SELECT MAX(data_lancamento) AS max_data 
    FROM gold.dim_movies 
    WHERE data_lancamento <= current_date()
)
SELECT c.nome_produtora, SUM(f.lucro_usd) AS lucro_total_usd
FROM gold.dim_companies c
JOIN gold.bridge_movie_company b ON c.sk_company_id = b.sk_company_id
JOIN gold.dim_movies d ON b.sk_movie_id = d.sk_movie_id
JOIN gold.fact_movies_performance f ON d.sk_movie_id = f.sk_movie_id
CROSS JOIN Limite l
WHERE d.data_lancamento >= add_months(l.max_data, -60)
  AND d.data_lancamento <= l.max_data
GROUP BY c.nome_produtora
ORDER BY lucro_total_usd DESC NULLS LAST
LIMIT 1;

nome_produtora,lucro_total_usd
Rkpix,6507052613.00
